# Neural Network as a Universal Function Approximator

The **universal approximation theorem** says that a neural network with at least one hidden layer and a
non-linear activation can approximate any continuous function as closely as we like.

In this notebook we test that claim with the custom library in this project (`DenseLayer`, `Tanh`, `MSE`, `train`, `predict`).
The network is trained to reproduce a 2D surface $z = f(x, y)$ using nothing but `(x, y) -> z` samples, and we record an
epoch-by-epoch animation:

* **left**: the original function (the target),
* **right**: the surface predicted by the network, with the original drawn faintly on top as a reference.

## 1. Imports and target function

In [1]:
import os
import sys

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.colors import LightSource, LinearSegmentedColormap, Normalize
from IPython.display import HTML, display

# Make the project root importable (Layer, Functions, Network) even when the notebook runs from Examples/
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "Examples" else os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

# ---- our own library ----
from Layer.DenseLayer import DenseLayer
from Functions.ActivationFunctions import Tanh
from Functions.LossFunctions import MSE
from Network.train import train
from Network.predict import predict

np.random.seed(42)   # reproducible weight initialisation and sample order

DOMAIN = 3.0         # x and y both live in [-DOMAIN, DOMAIN]


def target_function(x, y):
    """
    The function the network has to learn: the classic 'peaks' surface.
    Three hills/valleys plus a polynomial twist - smooth, but strongly non-linear.
        f(x,y) = 3(1-x)^2 e^{-x^2-(y+1)^2} - 10(x/5 - x^3 - y^5) e^{-x^2-y^2} - (1/3) e^{-(x+1)^2-y^2}
    Swap this for any other f(x, y) and the rest of the notebook still works.
    """
    return (3 * (1 - x) ** 2 * np.exp(-(x ** 2) - (y + 1) ** 2)
            - 10 * (x / 5 - x ** 3 - y ** 5) * np.exp(-(x ** 2) - y ** 2)
            - np.exp(-((x + 1) ** 2) - y ** 2) / 3)

## 2. Build the training data

In [2]:
GRID_SIZE = 32   # 32 x 32 = 1024 training points spread evenly over the domain

axis_vals = np.linspace(-DOMAIN, DOMAIN, GRID_SIZE)
X_grid, Y_grid = np.meshgrid(axis_vals, axis_vals)
Z_true = target_function(X_grid, Y_grid)          # the true height at every grid point

# Our library works on column vectors:  input -> (2, 1) = [x, y]^T,  target -> (1, 1) = [z]
num_samples = GRID_SIZE * GRID_SIZE
x_train = np.stack([X_grid.ravel(), Y_grid.ravel()], axis=1).reshape(num_samples, 2, 1)
y_train = Z_true.ravel().reshape(num_samples, 1, 1)

print(f"Training points : {num_samples} ({GRID_SIZE}x{GRID_SIZE} grid)")
print(f"x_train shape   : {x_train.shape}")
print(f"y_train shape   : {y_train.shape}")
print(f"z range         : [{Z_true.min():.2f}, {Z_true.max():.2f}]  (the output layer is linear, so no scaling is needed)")

Training points : 1024 (32x32 grid)
x_train shape   : (1024, 2, 1)
y_train shape   : (1024, 1, 1)
z range         : [-6.50, 7.98]  (the output layer is linear, so no scaling is needed)


## 3. Build the network and train it while recording the predicted surface

* Three hidden `Tanh` layers (2 → 64 → 64 → 32 → 1) give the network enough capacity for this surface.
* The last `DenseLayer` has **no activation** (linear output), so predictions are not squeezed into `[-1, 1]`.
* The loss is `MSE`. We call `train(..., epochs=1)` once per epoch so that we can take a snapshot of the surface after every epoch.

In [3]:
# ---------------- network ----------------
network = [
    DenseLayer(2, 64),  Tanh(),
    DenseLayer(64, 64), Tanh(),
    DenseLayer(64, 32), Tanh(),
    DenseLayer(32, 1),            # linear output
]
loss_fn = MSE()

# ---------------- hyper-parameters ----------------
EPOCHS = 720                         # total training epochs
LEARNING_RATE = 0.0004               # small step: z spans ~15 units, so this keeps training stable
SAMPLES_PER_EPOCH = num_samples // 4 # random quarter of the data per epoch (slows convergence -> nicer animation)
FRAME_EVERY = 2                      # store one animation frame every N epochs
FPS = 12                             # frames per second of the final GIF

# ---------------- fine grid used only for drawing ----------------
FINE = 72                            # 72 x 72 points -> smooth looking surface
fine_axis = np.linspace(-DOMAIN, DOMAIN, FINE)
Xf, Yf = np.meshgrid(fine_axis, fine_axis)
Z_true_f = target_function(Xf, Yf)   # exact target on the fine grid (no interpolation)
fine_points = np.stack([Xf.ravel(), Yf.ravel()], axis=0)        # (2, FINE*FINE)
train_points = np.stack([X_grid.ravel(), Y_grid.ravel()], axis=0)  # (2, num_samples)


def predict_points(points):
    """
    Run the network on many points at once.
    Our layers compute  W @ X + B  so a (2, N) matrix is simply N column vectors side by side.
    """
    return predict(network, points)   # shape (1, N)


def current_loss():
    """MSE of the network on the whole training grid."""
    return float(loss_fn.forward(predict_points(train_points), y_train.reshape(1, -1)))


# ---------------- recording buffers ----------------
frames_Z = [predict_points(fine_points).reshape(FINE, FINE)]   # epoch 0 = untrained network
frames_epoch = [0]
loss_history = [current_loss()]

print(f"Training for {EPOCHS} epochs...")
for epoch in range(1, EPOCHS + 1):
    # pick a random subset of the data for this epoch
    idx = np.random.permutation(num_samples)[:SAMPLES_PER_EPOCH]

    # one epoch of training with our own train(); plotting is off because we draw our own animation
    train(network, loss_fn, x_train[idx], y_train[idx],
          epochs=1, learning_rate=LEARNING_RATE, verbose=False, plot=False)

    loss_history.append(current_loss())

    # snapshot of the learned surface for the animation
    if epoch % FRAME_EVERY == 0 or epoch == EPOCHS:
        frames_Z.append(predict_points(fine_points).reshape(FINE, FINE))
        frames_epoch.append(epoch)

    if epoch == 1 or epoch % 60 == 0:
        print(f"  Epoch {epoch:4d}/{EPOCHS}  -  MSE loss: {loss_history[-1]:.6f}")

print("Training complete!")
print(f"Loss went from {loss_history[0]:.4f} to {loss_history[-1]:.5f}")

Training for 720 epochs...
  Epoch    1/720  -  MSE loss: 2.752604
  Epoch   60/720  -  MSE loss: 1.344832
  Epoch  120/720  -  MSE loss: 0.403469
  Epoch  180/720  -  MSE loss: 0.263972
  Epoch  240/720  -  MSE loss: 0.150101
  Epoch  300/720  -  MSE loss: 0.108256
  Epoch  360/720  -  MSE loss: 0.049709
  Epoch  420/720  -  MSE loss: 0.032064
  Epoch  480/720  -  MSE loss: 0.022462
  Epoch  540/720  -  MSE loss: 0.023082
  Epoch  600/720  -  MSE loss: 0.016638
  Epoch  660/720  -  MSE loss: 0.016382
  Epoch  720/720  -  MSE loss: 0.011510
Training complete!
Loss went from 3.4339 to 0.01151


## 4. Render the animation

Layout: **left** = original function, **right** = network prediction (updated every frame) with the original surface drawn
as a faint wireframe for reference, **bottom** = loss curve. The camera is fixed so only the surface changes.

In [4]:
# ---------------- colour theme (edit to restyle) ----------------
THEME = dict(
    bg="#000000",       # background
    panel="#0a0a0a",    # loss-plot background
    text="#ECECEC",     # main text
    grey="#888888",     # axes / grid
    blue="#58C4DD",     # network surface
    blue_d="#236B8E",   # low end of the colour map
    teal="#5CD0B3",     # high end of the colour map
    yellow="#FFFF00",   # original function / highlights
    red="#FC6255",      # loss curve
)
plt.rcdefaults()
plt.rcParams.update({
    "font.family": "serif",
    "mathtext.fontset": "cm",       # LaTeX-like maths font
    "text.color": THEME["text"],
    "axes.labelcolor": THEME["text"],
    "xtick.color": THEME["grey"],
    "ytick.color": THEME["grey"],
})
CMAP = LinearSegmentedColormap.from_list("height", [THEME["blue_d"], THEME["blue"], THEME["teal"]])

# ---------------- fixed camera + shared scales ----------------
ELEV, AZIM = 28, -58
Z_LO, Z_HI = float(Z_true_f.min()), float(Z_true_f.max())
Z_PAD = 0.12 * (Z_HI - Z_LO)
surface_norm = Normalize(vmin=Z_LO, vmax=Z_HI)      # same height->colour mapping on both plots, every frame
light = LightSource(azdeg=300, altdeg=40)           # soft light so the surface looks 3D
loss_floor, loss_ceil = min(loss_history) * 0.8, max(loss_history) * 1.3


def shaded(Z):
    """Colour every surface point by its height and add lighting."""
    return light.shade(Z, cmap=CMAP, norm=surface_norm, blend_mode="soft", vert_exag=1.2)


def network_surface(ax, Z):
    """Draw the network's surface and return the artist so the next frame can remove it."""
    Z = np.clip(Z, Z_LO - 3 * Z_PAD, Z_HI + 3 * Z_PAD)      # keep wild early predictions inside the box
    return ax.plot_surface(Xf, Yf, Z, facecolors=shaded(Z), shade=False, antialiased=True,
                           linewidth=0, rstride=1, cstride=1, zorder=1)


def style_3d(ax):
    """Same fixed camera and limits for both 3D plots."""
    ax.set_xlim(-DOMAIN, DOMAIN)
    ax.set_ylim(-DOMAIN, DOMAIN)
    ax.set_zlim(Z_LO - Z_PAD, Z_HI + Z_PAD)
    ax.set_box_aspect((1, 1, 0.75), zoom=1.15)
    ax.view_init(elev=ELEV, azim=AZIM)      # set once, never changed -> the camera does not move
    ax.set_axis_off()


fig = plt.figure(figsize=(16, 9), dpi=100, facecolor=THEME["bg"])

# title and formula of the target
fig.text(0.5, 0.965, "Neural Network as a Universal Function Approximator",
         ha="center", va="top", fontsize=26, color=THEME["text"])
fig.text(0.5, 0.905,
         r"$f(x,y)=3(1-x)^{2}e^{-x^{2}-(y+1)^{2}}-10\left(\frac{x}{5}-x^{3}-y^{5}\right)e^{-x^{2}-y^{2}}-\frac{1}{3}e^{-(x+1)^{2}-y^{2}}$",
         ha="center", va="top", fontsize=16, color=THEME["yellow"])

# LEFT: the original function (static, drawn once)
ax_true = fig.add_axes([0.00, 0.31, 0.50, 0.50], projection="3d", facecolor=THEME["bg"], computed_zorder=False)
ax_true.plot_surface(Xf, Yf, Z_true_f, facecolors=shaded(Z_true_f), shade=False, antialiased=True,
                     rstride=4, cstride=4, edgecolor=THEME["yellow"], linewidth=0.7, zorder=1)
style_3d(ax_true)
fig.text(0.25, 0.80, "Original function", ha="center", fontsize=20, color=THEME["yellow"])

# RIGHT: the network's prediction (replaced every frame) + the original as a faint reference mesh
ax_nn = fig.add_axes([0.50, 0.31, 0.50, 0.50], projection="3d", facecolor=THEME["bg"], computed_zorder=False)
ax_nn.plot_wireframe(Xf, Yf, Z_true_f, rstride=4, cstride=4,
                     color=THEME["yellow"], linewidth=0.6, alpha=0.3, zorder=5)   # low alpha = faint reference
style_3d(ax_nn)
fig.text(0.75, 0.80, "Neural network prediction", ha="center", fontsize=20, color=THEME["blue"])
nn_artist = [None]   # holds the surface drawn in the previous frame

# BOTTOM: loss curve on a log scale
ax_loss = fig.add_axes([0.07, 0.08, 0.88, 0.19], facecolor=THEME["panel"])
ax_loss.set_yscale("log")
ax_loss.set_xlim(0, EPOCHS)
ax_loss.set_ylim(loss_floor, loss_ceil)
ax_loss.set_xlabel("Epoch", fontsize=14)
ax_loss.set_ylabel("Loss (MSE)", fontsize=14)
ax_loss.grid(True, which="major", linewidth=0.5, color=THEME["grey"], alpha=0.25)
for spine in ax_loss.spines.values():
    spine.set_color(THEME["grey"])
(loss_line,) = ax_loss.plot([], [], color=THEME["red"], linewidth=2.8)
(loss_dot,) = ax_loss.plot([], [], "o", color=THEME["yellow"], markersize=9)
status_text = fig.text(0.5, 0.285, "", ha="center", va="center", fontsize=17, color=THEME["text"])


def update_frame(i):
    """Draw animation frame i."""
    epoch_num = frames_epoch[i]

    # replace only the network surface; everything else stays as it is
    if nn_artist[0] is not None:
        nn_artist[0].remove()
    nn_artist[0] = network_surface(ax_nn, frames_Z[i])

    # grow the loss curve up to the current epoch
    loss_line.set_data(np.arange(epoch_num + 1), loss_history[: epoch_num + 1])
    loss_dot.set_data([epoch_num], [loss_history[epoch_num]])
    status_text.set_text(f"Epoch {epoch_num:3d} / {EPOCHS}        Loss = {loss_history[epoch_num]:.5f}")

    if (i + 1) % 50 == 0:
        print(f"  Rendered frame {i + 1}/{len(frames_Z)}...")
    return ()


n_frames = len(frames_Z)
anim = FuncAnimation(fig, update_frame, frames=n_frames, interval=1000 // FPS, blit=False)

gif_path = "universal_approximator.gif"
print(f"Saving {n_frames} frames @ {FPS} FPS (~{n_frames / FPS:.0f}s) to '{gif_path}'...")
anim.save(gif_path, writer=PillowWriter(fps=FPS))
plt.close(fig)
print(f"Done! Saved animation to '{gif_path}'.")

# the query string forces the notebook to show the new GIF instead of a cached one
display(HTML(f'<img src="{gif_path}?v={__import__("time").time_ns()}" style="width:100%; height:auto;" />'))

Saving 361 frames @ 12 FPS (~30s) to 'universal_approximator.gif'...
  Rendered frame 50/361...
  Rendered frame 100/361...
  Rendered frame 150/361...
  Rendered frame 200/361...
  Rendered frame 250/361...
  Rendered frame 300/361...
  Rendered frame 350/361...
Done! Saved animation to 'universal_approximator.gif'.


## 5. How good is the fit?

In [5]:
# Compare the final network with the true function on the fine grid
Z_final = frames_Z[-1]
abs_err = np.abs(Z_final - Z_true_f)

print(f"Final training MSE : {loss_history[-1]:.5f}")
print(f"Mean absolute error: {abs_err.mean():.4f}   (surface height range is {Z_HI - Z_LO:.1f})")
print(f"Max absolute error : {abs_err.max():.4f}")

Final training MSE : 0.01151
Mean absolute error: 0.0742   (surface height range is 14.6)
Max absolute error : 0.6678
